In [ ]:
import time
start_time = time.time()
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
import spark_sdk as ss
import yaml
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import col, sum as sum_, count, max as max_, datediff, lit, min as min_, desc, when, unix_timestamp, row_number, struct, collect_list,\
to_date, countDistinct, exp, lit
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import pytz

import spark_sdk as ss
ss.PySpark(yarn=False, num_executors=48, driver_memory='32g', executor_memory='32g',
           add_on_config0=('spark.port.maxRetries', '1000'),
           add_on_config1=("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog"),
           add_on_config2=('spark.databricks.delta.retentionDurationCheck.enabled', 'false'),
           add_on_config3=('spark.sql.sources.partitionOverwriteMode', 'dynamic')
          )
spark = ss.PySpark().spark

from pyspark.sql.functions import to_date

full_dataset = spark.read.format('parquet').load('file:/home/baond12/projects/AI_Recommend_System_2025/BTL_HTTM/CS-recommendation-system/backend/full_dataset.parquet')\
.withColumn('event_date', to_date('TimePlaced'))

# Lấy danh sách 7 ngày cuối cùng có trong dữ liệu
max_date_row = full_dataset.agg(max_("event_date")).collect()[0][0]

evaluation_date_list = []
precision_at_10_list = []
recall_score_list = []
map_score_list = []
ndcg_at_10_list = []
hit_rate_score_list = []
coverage_score_list = []

for day_index in range(7):
    # Tính toán ngày bắt đầu validation (cách max date 6 ngày)
    val_date = max_date_row - timedelta(days=day_index)
    
    val_data = full_dataset.filter(f"event_date = '{val_date}'")
    
    print(f"Validation Date: {val_date}")
    
    # Load combo_features_recoms_daily
    combo_features_recoms_daily = spark.read.format('parquet').load("file:/mnt/projects-data/recommendation/dev_folder/baond12/btl_httm/combo_features_recoms_daily.parquet")\
    .where(f"d='{val_date.strftime('%Y-%m-%d')}'")
    
    # ====================================================
    # BƯỚC 1: CHUẨN BỊ GROUND TRUTH (Tập Validation)
    # ====================================================
    # Gom tất cả sản phẩm user thực sự mua trong ngày val thành 1 danh sách
    val_ground_truth = val_data.groupBy("Account_Id")\
        .agg(collect_list("ProductId").alias("true_items"))
    
    # ====================================================
    # BƯỚC 2: CHUẨN BỊ PREDICTION (Top 10 từ Model)
    # ====================================================
    # user_item_recs_final đang ở dạng long, cần lấy Top 10 score cao nhất
    # và gom lại thành list để so sánh
    preds_top_k = combo_features_recoms_daily.filter(col("rank") <= 10)\
        .groupBy("Account_Id")\
        .agg(collect_list("ProductId").alias("pred_items"))
    
    # ====================================================
    # BƯỚC 3: KẾT HỢP VÀ TÍNH TOÁN METRICS
    # ====================================================
    # Chỉ đánh giá trên những user CÓ trong tập Validation (để đo độ chính xác thực tế)
    # Join: [Account_Id, true_items, pred_items]
    evaluation_df = preds_top_k.join(val_ground_truth, on="Account_Id", how="inner")
    
    from pyspark.mllib.evaluation import RankingMetrics
    from pyspark.sql.functions import col, collect_list, size, array_intersect, expr, explode
    
    # --- A. Sử dụng Spark RankingMetrics (Cho MAP, NDCG, Precision) ---
    # RankingMetrics yêu cầu RDD của (prediction, ground_truth)
    prediction_and_labels = evaluation_df.rdd.map(lambda row: (row.pred_items, row.true_items))
    metrics_rdd = RankingMetrics(prediction_and_labels)
    
    precision_at_10 = metrics_rdd.precisionAt(10)
    ndcg_at_10 = metrics_rdd.ndcgAt(10)
    map_score = metrics_rdd.meanAveragePrecision
    
    # --- B. Tính toán thủ công (Cho Recall, HitRate, Coverage) ---
    
    # 1. Tính các chỉ số chi tiết từng dòng (Row-wise)
    # Hit: Nếu giao nhau giữa pred và true > 0 thì là hit
    # Recall: Số item đoán đúng / Tổng số item user đã mua
    eval_stats = evaluation_df.withColumn("hits_count", size(array_intersect(col("pred_items"), col("true_items"))))\
        .withColumn("true_count", size(col("true_items")))\
        .withColumn("is_hit", when(col("hits_count") > 0, 1).otherwise(0))\
        .withColumn("recall", col("hits_count") / col("true_count"))
    
    # Aggregation để lấy trung bình
    agg_metrics = eval_stats.agg(
        (sum_(col("recall")) / count("*")).alias("avg_recall"),
        (sum_(col("is_hit")) / count("*")).alias("hit_rate")
    ).collect()[0]
    
    recall_score = agg_metrics["avg_recall"]
    hit_rate_score = agg_metrics["hit_rate"]
    
    # 2. Tính Coverage (Item Coverage)
    # Tỷ lệ % sản phẩm được recommend trên tổng số sản phẩm có trong training
    # Đếm số lượng unique items trong tập Prediction (preds_top_k)
    unique_recommended_items = preds_top_k.select(explode("pred_items").alias("item")).select("item").distinct().count()
    total_items_in_train = combo_features_recoms_daily.select("ProductId").distinct().count()
    
    coverage_score = unique_recommended_items / total_items_in_train if total_items_in_train > 0 else 0
    
    # ====================================================
    # BƯỚC 4: IN KẾT QUẢ
    # ====================================================
    print("\n" + "="*40)
    print(f"REPORT EVALUATION @ K=10 (Date: {val_date})")
    print("="*40)
    print(f"Precision@10 : {precision_at_10:.4f}")
    print(f"Recall@10    : {recall_score:.4f}")
    print(f"MAP          : {map_score:.4f}")
    print(f"NDCG@10      : {ndcg_at_10:.4f}")
    print(f"Hit Rate     : {hit_rate_score:.4f} (Tỷ lệ user tìm thấy ít nhất 1 sp ưng ý)")
    print(f"Coverage     : {coverage_score:.4f} ({unique_recommended_items}/{total_items_in_train} items)")
    print("="*40)
    evaluation_date_list.append(val_date.strftime('%Y-%m-%d'))
    precision_at_10_list.append(precision_at_10)
    recall_score_list.append(recall_score)
    map_score_list.append(map_score)
    ndcg_at_10_list.append(ndcg_at_10)
    hit_rate_score_list.append(hit_rate_score)
    coverage_score_list.append(coverage_score)

final_report_df = pd.DataFrame({
    'evaluation_date': evaluation_date_list, 
    'precision_at_10': precision_at_10_list, 
    'recall_score': recall_score_list, 
    'map_score': map_score_list, 
    'ndcg_at_10': ndcg_at_10_list, 
    'hit_rate_score': hit_rate_score_list, 
    'coverage_score': coverage_score_list
})

final_report_df.to_excel('./benchmarks_by_evaluation_dates.xlsx', index=False)